# Pipeline RAG — Recherche vectorielle, FAISS, ChromaDB & Question-Answering

Ce notebook construit un pipeline **RAG (Retrieval-Augmented Generation)** complet :
vectorisation de titres d'actualités, stockage/recherche vectorielle avec **FAISS** et **ChromaDB**,
recherche de similarité, puis réponse aux questions avec un LLM **Hugging Face**.

**Recommandé : exécuter sur Google Colab avec un runtime GPU** (Runtime → Change runtime type → GPU).

## 🌟 Exercice 1 — Chargement et préparation des données

### 1. Installation des bibliothèques

> Sur Colab, exécutez la cellule ci-dessous. Les versions sont épinglées comme dans l'énoncé pour éviter les conflits d'API.

In [ ]:
%%capture
!pip install -q faiss-cpu==1.7.4
!pip install -q chromadb==0.3.21
!pip install -qU sentence-transformers
!pip install -q "numpy<2"
!apt-get -qq install -y libomp-dev

In [ ]:
import os
os.makedirs("cache", exist_ok=True)   # répertoire de cache pour les fichiers intermédiaires
print("Dossier 'cache' prêt :", os.path.isdir("cache"))

In [ ]:
import numpy as np
import pandas as pd
import faiss
import json

from sentence_transformers import SentenceTransformer, InputExample
import chromadb
from chromadb.config import Settings
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

print("Imports OK")

**Rôle des bibliothèques :**
- `faiss-cpu` : recherche de similarité et clustering efficaces de vecteurs denses (Facebook AI Research).
- `chromadb` : base de données vectorielle pour stocker et interroger des embeddings.
- `sentence-transformers` : génération d'embeddings de phrases.
- `transformers` : modèles de langage pré-entraînés (Hugging Face).

### 2. Charger le jeu de données

Le fichier `labelled_newscatcher_dataset.csv` contient des articles de presse étiquetés (séparateur `;`).
On le charge ici directement depuis le dépôt GitHub officiel du dataset.

In [ ]:
# Option A — chargement direct depuis GitHub (recommandé, rien à téléverser)
path = "https://raw.githubusercontent.com/kotartemiy/topic-labeled-news-dataset/master/labeled_newscatcher_dataset.csv"

# Option B — si vous avez téléversé le fichier sur Colab, décommentez :
# path = "labelled_newscatcher_dataset.csv"

pdf = pd.read_csv(path, sep=";")
print("Dimensions :", pdf.shape)

### 3. Ajouter une colonne d'identifiant unique

In [ ]:
pdf["id"] = pdf.index
pdf.head()

### 4. Analyser les données

On observe les colonnes, les types et les valeurs manquantes.

In [ ]:
display(pdf.head(10))
print("\nColonnes :", list(pdf.columns))
print("\nTypes :\n", pdf.dtypes)
print("\nValeurs manquantes par colonne :\n", pdf.isnull().sum())
print("\nRépartition des sujets (topic) :\n", pdf["topic"].value_counts())

**Observations :** le dataset contient notamment les colonnes `topic` (étiquette de sujet),
`title` (titre de l'article, notre texte à vectoriser), `link`, `domain`, `published_date` et `lang`.
Les articles couvrent 8 sujets (BUSINESS, ENTERTAINMENT, HEALTH, NATION, SCIENCE, SPORTS, TECHNOLOGY, WORLD).

### 5. Créer un sous-ensemble pour un traitement plus rapide

On garde les 1000 premières lignes pour itérer vite pendant le développement.

In [ ]:
pdf_subset = pdf.head(1000).copy()
print("Sous-ensemble :", pdf_subset.shape)

## 🌟 Exercice 2 — Vectorisation avec Sentence Transformers

### 1–3. Fonction auxiliaire `example_create_fn`

Elle formate chaque titre en un objet `InputExample` (attendu par Sentence Transformers).

In [ ]:
from sentence_transformers import InputExample

def example_create_fn(doc1: pd.Series) -> InputExample:
    """Formate un titre d'article en objet InputExample (guid, label, texts)."""
    return InputExample(texts=[doc1])

### 4. Appliquer la fonction auxiliaire au sous-ensemble

In [ ]:
faiss_train_examples = pdf_subset.apply(
    lambda x: example_create_fn(x["title"]), axis=1
).tolist()

faiss_train_examples[:10]

### 5. Initialiser le modèle d'embedding `all-MiniLM-L6-v2`

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")
print("Modèle chargé. Dimension des embeddings :", model.get_sentence_embedding_dimension())

### 6. Extraire les titres sous forme de liste de chaînes

In [ ]:
titles_list = pdf_subset["title"].tolist()
print("Nombre de titres :", len(titles_list))
print("Exemple :", titles_list[0])

### 7. Générer les embeddings des titres

In [ ]:
faiss_title_embedding = model.encode(titles_list, show_progress_bar=True)
print("Type :", type(faiss_title_embedding))

### 8. Vérifier les dimensions des embeddings

In [ ]:
len(faiss_title_embedding), len(faiss_title_embedding[0])

On obtient **1000 embeddings** (un par titre), chacun de **384 dimensions** (dimension propre à `all-MiniLM-L6-v2`).

## 🌟 Exercice 3 — Indexation et recherche FAISS

In [ ]:
import numpy as np
import faiss

### 2. Préparer les données à indexer

In [ ]:
pdf_to_index = pdf_subset.copy()
id_index = np.array(pdf_to_index["id"], dtype="int64")
print("IDs à indexer :", id_index[:10], "...")

### 3. Normaliser les embeddings (pour la similarité cosinus)

Avec des vecteurs normalisés (longueur unitaire), le **produit scalaire = similarité cosinus**.

In [ ]:
content_encoded_normalized = faiss_title_embedding.copy().astype("float32")
faiss.normalize_L2(content_encoded_normalized)
print("Vecteurs normalisés :", content_encoded_normalized.shape)

### 4. Créer l'index FAISS (`IndexFlatIP` encapsulé dans `IndexIDMap`)

In [ ]:
index_content = faiss.IndexIDMap(faiss.IndexFlatIP(len(faiss_title_embedding[0])))
index_content.add_with_ids(content_encoded_normalized, id_index)
print("Nombre de vecteurs dans l'index :", index_content.ntotal)

- `IndexFlatIP` : recherche exacte par produit scalaire (= cosinus sur vecteurs normalisés).
- `IndexIDMap` : associe chaque vecteur à son identifiant d'origine, pour retrouver l'article correspondant.

### 5. Fonction de recherche `search_content`

In [ ]:
def search_content(query, pdf_to_index, k=3):
    # Encoder la requête et la normaliser
    query_vector = model.encode([query]).astype("float32")
    faiss.normalize_L2(query_vector)

    # Recherche des k plus proches voisins
    top_k = index_content.search(query_vector, k)
    ids = top_k[1][0].tolist()            # identifiants des vecteurs correspondants
    similarities = top_k[0][0].tolist()   # scores de similarité (cosinus)

    # Récupérer les articles correspondants
    results = pdf_to_index.loc[ids].copy()
    results["similarities"] = similarities
    return results

### 6. Tester la recherche

In [ ]:
display(search_content("animal", pdf_to_index, k=5))

## 🌟 Exercice 4 — Collection et requêtes ChromaDB

Contrairement à FAISS, **ChromaDB gère automatiquement** la tokenisation, l'embedding et l'indexation
(via `SentenceTransformerEmbeddingFunction` par défaut).

In [ ]:
import chromadb
from chromadb.config import Settings

### 2. Initialiser le client et créer une collection

In [ ]:
chroma_client = chromadb.Client()
collection_name = "my_news"

# Supprimer la collection si elle existe déjà (évite les conflits)
if len(chroma_client.list_collections()) > 0 and collection_name in [chroma_client.list_collections()[0].name]:
    chroma_client.delete_collection(name=collection_name)

print(f"Creating collection: '{collection_name}'")
collection = chroma_client.create_collection(name=collection_name)

### 3. Ajouter les 100 premiers titres (avec sujet en métadonnée et ID unique)

In [ ]:
display(pdf_subset.head())

collection.add(
    documents=pdf_subset["title"][:100].tolist(),
    metadatas=[{"topic": topic} for topic in pdf_subset["topic"][:100].tolist()],
    ids=[str(i) for i in pdf_subset["id"][:100].tolist()],   # identifiants uniques (str)
)
print("Documents ajoutés :", collection.count())

### 4. Interroger la collection (10 documents les plus pertinents)

In [ ]:
import json

results = collection.query(
    query_texts=["space"],
    n_results=10,
)

print(json.dumps(results, indent=4))

ChromaDB convertit automatiquement le terme « space » en embedding, puis renvoie les 10 voisins
les plus proches (documents, métadonnées et distances).

## 🌟 Exercice 5 — Question-Answering avec un modèle Hugging Face

On combine la **récupération** (ChromaDB) et la **génération** (Hugging Face) → pipeline RAG.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

### 2. Initialiser le modèle et le tokenizer (GPT-2)

In [ ]:
model_id = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_id)
lm_model = AutoModelForCausalLM.from_pretrained(model_id)
print("Modèle de génération chargé :", model_id)

### 3. Créer le pipeline de génération de texte

In [ ]:
pipe = pipeline(
    "text-generation",
    model=lm_model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    device_map="auto",
)

### 4. Construire le prompt (contexte récupéré + question)

In [ ]:
question = "What's the latest news on space development?"

# Concaténer les documents récupérés par ChromaDB comme contexte
context = " ".join([f"#{str(i)}" for i in results["documents"][0]])

prompt_template = f"Relevant context: {context}\n\n The user's question: {question}"
print(prompt_template[:1000])

### 5. Générer une réponse

In [ ]:
lm_response = pipe(prompt_template)
print(lm_response[0]["generated_text"])

### 6. Expérimenter avec différentes questions et tailles de contexte

Faites varier la requête ChromaDB, le nombre de documents récupérés (`n_results`) et la question
pour observer l'effet sur la réponse générée.

In [ ]:
# Exemple : nouvelle requête + contexte réduit à 5 documents
results2 = collection.query(query_texts=["sports"], n_results=5)
question2 = "What are the recent sports headlines?"
context2 = " ".join([f"#{str(i)}" for i in results2["documents"][0]])
prompt2 = f"Relevant context: {context2}\n\n The user's question: {question2}"

lm_response2 = pipe(prompt2)
print(lm_response2[0]["generated_text"])

## Conclusion

Ce notebook a couvert un pipeline RAG complet :
1. **Préparation des données** (chargement, IDs, sous-ensemble).
2. **Vectorisation** des titres avec `all-MiniLM-L6-v2` (embeddings 384-D).
3. **FAISS** : index `IndexFlatIP` + `IndexIDMap`, recherche par similarité cosinus.
4. **ChromaDB** : collection avec embeddings automatiques, requêtes sémantiques + métadonnées.
5. **Question-Answering** : contexte récupéré injecté dans un LLM Hugging Face (GPT-2).

**Pistes d'amélioration :** utiliser un LLM instruct plus performant (ex. Flan-T5, Mistral),
augmenter le sous-ensemble, filtrer par métadonnée `topic`, et affiner le prompt pour de
meilleures réponses.